In [0]:
%pip install -qqqq -U mlflow-skinny[databricks] databricks-sdk databricks-openai "psycopg[binary]>=3.1.0" openai-agents databricks-mcp
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
default_warehouse = next(
    (
        wh
        for wh in w.warehouses.list()
        if "Serverless Starter Warehouse" in wh.name and wh.enable_serverless_compute
    ),
    None,
)
default_warehouse_id = default_warehouse.id if default_warehouse else None
print(f"{default_warehouse_id=}")

In [0]:
import mlflow
import os
from databricks_openai import DatabricksOpenAI

os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = default_warehouse_id
# Enable MLflow's autologging to instrument your application with Tracing
mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
EXPERIMENT_NAME = "/Workspace/Shared/bo-uc-eval-traces-code-test"
mlflow.set_experiment(EXPERIMENT_NAME)

# Create an OpenAI client that is connected to Databricks-hosted LLMs
client = DatabricksOpenAI()

# Select an LLM
model_name = "databricks-claude-opus-5"

In [0]:
import psycopg
from databricks.sdk import WorkspaceClient
from mlflow.entities import Feedback

# Lakebase connection config for the code-search project
LAKEBASE_PROJECT = "code-search"
LAKEBASE_BRANCH = "production"
LAKEBASE_ENDPOINT = "primary"
LAKEBASE_DB = "databricks_postgres"


def _search_code_bm25(query: str, top_k: int = 5) -> str:
    """Perform BM25 keyword search against the Lakebase code-search project."""
    ws = WorkspaceClient()
    endpoint_name = f"projects/{LAKEBASE_PROJECT}/branches/{LAKEBASE_BRANCH}/endpoints/{LAKEBASE_ENDPOINT}"
    endpoint = ws.postgres.get_endpoint(name=endpoint_name)
    host = endpoint.status.hosts.host
    user = ws.current_user.me().user_name
    token = ws.postgres.generate_database_credential(endpoint=endpoint_name).token

    sql = """
        SELECT c.content, f.path, r.name AS repo_name, r.default_branch,
               c.start_line, c.end_line,
               c.ts <@> to_bm25query(to_tsvector('english', %(query)s), 'ix_chunks_ts_bm25'::regclass) AS score
        FROM chunks c
        JOIN files f ON f.id = c.file_id
        JOIN repos r ON r.id = f.repo_id
        ORDER BY c.ts <@> to_bm25query(to_tsvector('english', %(query)s), 'ix_chunks_ts_bm25'::regclass)
        LIMIT %(top_k)s
    """

    with psycopg.connect(
        host=host, dbname=LAKEBASE_DB, user=user, password=token, sslmode="require"
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, {"query": query, "top_k": top_k})
            rows = cur.fetchall()

    if not rows:
        return "No relevant code found."

    snippets = []
    for content, path, repo_name, branch, start_line, end_line, score in rows:
        url = f"https://github.com/{repo_name}/blob/{branch}/{path}#L{start_line}-L{end_line}"
        snippets.append(
            f"--- {repo_name}: {path} (score: {score:.4f}) ---\n{url}\n{content}"
        )
    return "\n\n".join(snippets)

In [0]:
from typing import Dict

# Example questions about Databricks code patterns (grounded in indexed repos)
GENERIC_QUESTIONS = [
    "Show me how to import DatabricksOpenAI from databricks_openai and use it with WorkspaceClient to call chat.completions.create",
    # "what are Databricks security best practices",
    # "What is Databricks used for?",
    # "How does Databricks integrate with Apache Spark?",
    # "What are the benefits of using Databricks notebooks?",
    # "Can Databricks be used for machine learning?",
    # "How does Databricks handle data security?",
]


@mlflow.trace
def answer_databricks_question(question: str) -> Dict[str, str]:
    """Generate a code-grounded answer using BM25 retrieval from Lakebase."""
    # Retrieve relevant code snippets first (RAG)
    try:
        code_context = _search_code_bm25(question, top_k=3)
    except Exception:
        code_context = "No code context available."

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a Databricks code assistant. Answer questions based ONLY on "
                    "the provided code snippets. Do not invent APIs or patterns not shown "
                    "in the code. If the code shows a specific import or usage pattern, "
                    "use exactly that pattern in your answer."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    f"Reference code:\n{code_context}"
                ),
            },
        ],
        max_tokens=1000,
    )

    content = response.choices[0].message.content
    print(content)
    # Extract text from content blocks if it's a list (e.g., reasoning + text blocks)
    if isinstance(content, list):
        content = "\n".join(
            (
                block.get("text", "")
                if isinstance(block, dict) and block.get("type") == "text"
                else ""
            )
            for block in content
        ).strip()
    return {"answer": content}


# Test the application
for q in GENERIC_QUESTIONS:
    result = answer_databricks_question(q)
    print(f"Q: {q}\nA: {result['answer']}\n")

In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
traces_df = mlflow.search_traces(
    locations=[experiment.experiment_id],
    order_by=["timestamp DESC"],
    max_results=1,
)
print(f"Found {len(traces_df)} traces to evaluate")

In [0]:
from mlflow.genai.scorers import scorer


@scorer
def code_grounded(inputs, outputs) -> Feedback:
    """Score whether the response is grounded in actual code from BM25 search."""
    question = inputs.get("query", inputs.get("question", str(inputs)))

    # Retrieve relevant code snippets via Lakebase BM25 search
    try:
        code_snippets = _search_code_bm25(question)
    except Exception as e:
        return Feedback(
            value=False,
            rationale=f"Code search unavailable: {type(e).__name__}: {e}",
        )

    if code_snippets == "No relevant code found.":
        return Feedback(
            value=False,
            rationale="No relevant code snippets found for this question.",
        )

    # Use LLM to judge if the output is consistent with the retrieved code
    judge_response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an evaluation judge. Given a question, an answer, and "
                    "reference code snippets, determine whether the answer is factually "
                    "grounded in the code snippets. Respond with ONLY 'yes' or 'no' on "
                    "the first line, followed by a brief rationale on the next line."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    f"Answer: {outputs}\n\n"
                    f"Reference code snippets:\n{code_snippets}"
                ),
            },
        ],
        max_tokens=300,
    )

    content = judge_response.choices[0].message.content
    # Handle content as list of blocks (newer API format) or plain string
    if isinstance(content, list):
        content = "".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in content
        )
    judge_text = content.strip()
    lines = judge_text.split("\n", 1)
    verdict = lines[0].strip().lower() == "yes"
    rationale = lines[1].strip() if len(lines) > 1 else judge_text

    return Feedback(
        value=verdict,
        rationale=f"{rationale}\n\n--- Retrieved Code Snippets ---\n{code_snippets}",
    )

In [0]:
# import httpx
# import json
# from mlflow.genai.scorers import scorer

# GITHUB_PAT = dbutils.secrets.get(
#     catalog="workshop_guy_catalog", schema="genie_traces_bo", key="github_pat"
# )
# GITHUB_REPO = "bcheng004/databricks-genie-agents-mlflow"


# def _search_code_github(query: str, top_k: int = 5) -> str:
#     """Search code on GitHub via the REST API (PAT from Databricks Secrets)."""
#     headers = {
#         "Authorization": f"Bearer {GITHUB_PAT}",
#         "Accept": "application/vnd.github+json",
#         "X-GitHub-Api-Version": "2022-11-28",
#     }
#     resp = httpx.get(
#         "https://api.github.com/search/code",
#         params={"q": f"repo:{GITHUB_REPO} {query}", "per_page": top_k},
#         headers=headers,
#         timeout=30,
#     )
#     resp.raise_for_status()
#     data = resp.json()

#     if not data.get("items"):
#         return "No relevant code found on GitHub."

#     snippets = []
#     for item in data["items"][:top_k]:
#         repo = item["repository"]["full_name"]
#         path = item["path"]
#         url = item["html_url"]
#         # Fetch raw file content (first 80 lines)
#         try:
#             raw_resp = httpx.get(
#                 f"https://raw.githubusercontent.com/{repo}/HEAD/{path}",
#                 timeout=10,
#             )
#             content = "\n".join(raw_resp.text.splitlines()[:80])
#         except Exception:
#             content = "(content unavailable)"
#         snippets.append(f"--- {repo}: {path} ---\n{url}\n{content}")

#     return "\n\n".join(snippets)


# def _parse_snippet_paths(snippets_text: str) -> list:
#     """Extract repo, path, URL, and line numbers from the formatted snippet blocks."""
#     import re

#     results = []
#     for line in snippets_text.split("\n"):
#         if line.startswith("--- ") and line.endswith(" ---"):
#             inner = line[4:-4]
#             if ": " in inner:
#                 repo, path = inner.split(": ", 1)
#                 # Strip trailing score info if present
#                 path = (re.sub(r"\s*score:.*", "", path),)
#                 results.append({"repo": repo, "path": path, "url": "", "lines": ""})
#         elif line.startswith("http") and results and not results[-1]["url"]:
#             url = line.strip()
#             results[-1]["url"] = url
#             # Extract line numbers from URL fragment like #L10-L25
#             m = re.search(r"#L(\d+)(?:-L(\d+))?", url)
#             if m:
#                 start = m.group(1)
#                 end = m.group(2)
#                 results[-1]["lines"] = f"L{start}-L{end}" if end else f"L{start}"
#     return results


# @scorer
# def code_grounded_github(inputs, outputs) -> Feedback:
#     """Score whether the response is grounded in code found via GitHub search."""
#     question = inputs.get("query", inputs.get("question", str(inputs)))

#     # Retrieve relevant code snippets via GitHub REST API
#     try:
#         code_snippets = _search_code_github(question)
#     except Exception as e:
#         return Feedback(
#             value=False,
#             rationale=f"GitHub search unavailable: {type(e).__name__}: {e}",
#         )

#     if code_snippets == "No relevant code found on GitHub.":
#         return Feedback(
#             value=False,
#             rationale="No relevant code found via GitHub search.",
#         )

#     # Collect source paths for structured evidence
#     source_paths = []
#     for item in _parse_snippet_paths(code_snippets):
#         source_paths.append(item)

#     # Use LLM to judge groundedness with a detailed rationale
#     judge_response = client.chat.completions.create(
#         model=model_name,
#         messages=[
#             {
#                 "role": "system",
#                 "content": (
#                     "You are an evaluation judge. Given a question, an answer, and "
#                     "reference code snippets from GitHub, determine whether the answer is "
#                     "factually grounded in the code.\n\n"
#                     "Respond in this exact format:\n"
#                     "VERDICT: yes|no\n"
#                     "CONFIDENCE: high|medium|low\n"
#                     "RATIONALE: A 2-4 sentence explanation of why the answer is or is not "
#                     "grounded. Reference specific function names, imports, or patterns by "
#                     "name — do NOT reproduce code blocks or quote source lines verbatim."
#                 ),
#             },
#             {
#                 "role": "user",
#                 "content": (
#                     f"Question: {question}\n\n"
#                     f"Answer to evaluate:\n{outputs}\n\n"
#                     f"GitHub code snippets:\n{code_snippets}"
#                 ),
#             },
#         ],
#         max_tokens=600,
#     )

#     content = judge_response.choices[0].message.content
#     if isinstance(content, list):
#         content = "".join(
#             block.get("text", "") if isinstance(block, dict) else str(block)
#             for block in content
#         )
#     judge_text = content.strip()

#     # Parse structured response — flexible matching
#     verdict = any(
#         "yes" in line.lower()
#         for line in judge_text.split("\n")[:3]
#         if "verdict" in line.lower() or line.strip().lower() in ("yes", "no")
#     )
#     confidence = "unknown"
#     for line in judge_text.split("\n"):
#         low = line.strip().lower()
#         if "confidence" in low:
#             for level in ("high", "medium", "low"):
#                 if level in low:
#                     confidence = level
#                     break
#             break

#     # Extract just VERDICT/CONFIDENCE/RATIONALE lines (skip EVIDENCE or code)
#     rationale_lines = []
#     for line in judge_text.split("\n"):
#         stripped = line.strip()
#         if stripped.upper().startswith(("VERDICT:", "CONFIDENCE:", "RATIONALE:")):
#             rationale_lines.append(stripped)
#         elif rationale_lines and rationale_lines[-1].upper().startswith("RATIONALE:"):
#             # Continuation of RATIONALE (multi-line)
#             if not stripped.upper().startswith("EVIDENCE") and stripped:
#                 rationale_lines.append(stripped)
#             else:
#                 break

#     judge_summary = (
#         "\n".join(rationale_lines) if rationale_lines else judge_text.split("\n")[0]
#     )

#     # Build structured rationale: LLM reasoning + source references (no raw code)
#     structured_rationale = (
#         f"## LLM Judge Assessment (confidence: {confidence})\n\n"
#         f"{judge_summary}\n\n"
#         f"## Source References\n"
#     )
#     for src in source_paths:
#         line_info = f" ({src['lines']})" if src["lines"] else ""
#         structured_rationale += (
#             f"- [{src['repo']}: {src['path']}{line_info}]({src['url']})\n"
#         )

#     return Feedback(
#         value=verdict,
#         rationale=structured_rationale,
#     )

In [0]:
eval_results = mlflow.genai.evaluate(
    data=traces_df,
    scorers=[
        code_grounded,
        # code_grounded_github,
    ],
)

In [0]:
print("Metrics:", eval_results.metrics)
print()
for name, df in eval_results.tables.items():
    for _, row in df.iterrows():
        for col in df.columns:
            if "/value" in col:
                scorer_name = col.replace("/value", "")
                val = row[col]
                rat = row.get(f"{scorer_name}/rationale", "")
                print(f"\n{'='*60}")
                print(f"{scorer_name}: {val}")
                print(f"{'='*60}")
                print(f"Rationale (first 600 chars):\n{str(rat)[:600]}\n")